In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:

# ============================================================
# JFAMP — MobileNetV2 + CIFAR-100
# STAGE 0, CELL 1: Workspace, Dataset and Reproducibility Setup
# ============================================================

import os
import sys
import json
import time
import random
import hashlib
import platform
from pathlib import Path

import numpy as np
import torch
import torchvision
from torchvision import datasets

# ---------------- CONFIGURATION ----------------

SEED = 42

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
DATA_ROOT = ROOT / "data" / "torchvision"

# ---------------- REPRODUCIBILITY ----------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---------------- DIRECTORY STRUCTURE ----------------

directories = [
    ROOT / "checkpoints",
    ROOT / "logs",
    ROOT / "data",
    ROOT / "config",
    ROOT / "experiments",
    ROOT / "reports",
    ROOT / "backups",
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

for i in range(1, 9):
    (ROOT / "experiments" / f"experiment_{i:02d}").mkdir(
        parents=True, exist_ok=True
    )

# ---------------- DATASET DOWNLOAD ----------------

print("=" * 65)
print("JFAMP MOBILENETV2 — STAGE 0")
print("=" * 65)

print("\nDownloading/verifying CIFAR-100...")

train_dataset = datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=True
)

test_dataset = datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=False,
    download=True
)

assert len(train_dataset) == 50000
assert len(test_dataset) == 10000

print("Training archive:", len(train_dataset))
print("Official test archive:", len(test_dataset))

# ---------------- FIXED TRAIN / VALIDATION SPLIT ----------------

rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_dataset))

train_indices = indices[:45000].tolist()
validation_indices = indices[45000:].tolist()

assert len(train_indices) == 45000
assert len(validation_indices) == 5000
assert not set(train_indices).intersection(validation_indices)

split_data = {
    "dataset": "CIFAR-100",
    "architecture": "MobileNetV2",
    "seed": SEED,
    "total_training_archive": 50000,
    "train_size": 45000,
    "validation_size": 5000,
    "test_size": 10000,
    "train_indices": train_indices,
    "validation_indices": validation_indices,
    "split_method": "NumPy default_rng(seed=42).permutation"
}

split_path = ROOT / "data" / "data_split_indices.json"

with open(split_path, "w") as f:
    json.dump(split_data, f, indent=2)

# ---------------- HASHING ----------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_hash = sha256_file(split_path)

# ---------------- ENVIRONMENT MANIFEST ----------------

environment = {
    "project": "JFAMP",
    "architecture": "MobileNetV2",
    "dataset": "CIFAR-100",
    "seed": SEED,
    "python_version": sys.version,
    "platform": platform.platform(),
    "pytorch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "split_sha256": split_hash,
    "normalization": {
        "mean": [0.5071, 0.4867, 0.4408],
        "std": [0.2675, 0.2565, 0.2761]
    },
    "split_sizes": {
        "train": 45000,
        "validation": 5000,
        "official_test": 10000
    }
}

environment_path = ROOT / "config" / "environment.json"

with open(environment_path, "w") as f:
    json.dump(environment, f, indent=2)

# ---------------- DATASET MANIFEST ----------------

dataset_manifest = {
    "dataset": "CIFAR-100",
    "download_root": str(DATA_ROOT),
    "training_samples": len(train_dataset),
    "official_test_samples": len(test_dataset),
    "split_file": str(split_path),
    "split_sha256": split_hash,
    "download_verified": True
}

with open(ROOT / "data" / "dataset_manifest.json", "w") as f:
    json.dump(dataset_manifest, f, indent=2)

# ---------------- FINAL CHECKS ----------------

assert split_path.exists()
assert len(json.load(open(split_path))["train_indices"]) == 45000

print("\n" + "=" * 65)
print("STAGE 0 SETUP COMPLETED")
print("=" * 65)

print("Workspace:", ROOT)
print("Dataset root:", DATA_ROOT)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_indices))
print("Official test samples:", len(test_dataset))
print("Random seed:", SEED)
print("Split SHA256:", split_hash)

print("\nGPU:", environment["gpu"])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("\nSaved artifacts:")
print(" - data/data_split_indices.json")
print(" - data/dataset_manifest.json")
print(" - config/environment.json")

print("\nDirectory structure:")
for p in sorted(ROOT.iterdir()):
    print(" ", p.name + ("/" if p.is_dir() else ""))

print("\nNEXT: MobileNetV2 model training.")


JFAMP MOBILENETV2 — STAGE 0

Downloading/verifying CIFAR-100...


100%|██████████| 169M/169M [24:11<00:00, 116kB/s]  


Training archive: 50000
Official test archive: 10000

STAGE 0 SETUP COMPLETED
Workspace: /kaggle/working/JFAMP_MobileNetV2
Dataset root: /kaggle/working/JFAMP_MobileNetV2/data/torchvision
Training samples: 50000
Validation samples: 5000
Official test samples: 10000
Random seed: 42
Split SHA256: e7eaa993c06cb2f5eb9aa8ce111bb8e78ad7a87aa369f04a272e773d23a1039b

GPU: Tesla T4
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130

Saved artifacts:
 - data/data_split_indices.json
 - data/dataset_manifest.json
 - config/environment.json

Directory structure:
  backups/
  checkpoints/
  config/
  data/
  experiments/
  logs/
  reports/

NEXT: MobileNetV2 model training.


In [7]:

# ================================================================
# JFAMP MOBILENETV2 — STAGE 1: MODEL TRAINING
# CIFAR-100 | 150 epochs | Seed 42 | Tesla T4
# ================================================================

import os
import json
import time
import random
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision.models import mobilenet_v2

# -------------------------
# 1. Configuration
# -------------------------

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
DATA_ROOT = ROOT / "data" / "torchvision"
CHECKPOINT_DIR = ROOT / "checkpoints"
LOG_DIR = ROOT / "logs"
CONFIG_DIR = ROOT / "config"

BEST_PATH = CHECKPOINT_DIR / "mobilenetv2_cifar100_best.pth"
LATEST_PATH = CHECKPOINT_DIR / "mobilenetv2_cifar100_latest.pth"
HISTORY_PATH = LOG_DIR / "training_history.csv"

SEED = 42
EPOCHS = 150
BATCH_SIZE = 128
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# 2. Reproducibility
# -------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -------------------------
# 3. Data transforms
# -------------------------

MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# Separate dataset objects ensure validation has no augmentation.
train_dataset_full = CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=train_transform
)

val_dataset_full = CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=eval_transform
)

official_test_dataset = CIFAR100(
    root=str(DATA_ROOT),
    train=False,
    download=False,
    transform=eval_transform
)

with open(ROOT / "data" / "data_split_indices.json", "r") as f:
    split = json.load(f)

train_indices = split["train_indices"]
val_indices = split["validation_indices"]

assert len(train_indices) == 45000
assert len(val_indices) == 5000
assert len(set(train_indices).intersection(val_indices)) == 0

print("Split verified successfully.")
print("Training samples:", len(train_indices))
print("Validation samples:", len(val_indices))

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    official_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Dataset loaders ready:")
print("  Training:", len(train_dataset))
print("  Validation:", len(val_dataset))
print("  Official test:", len(official_test_dataset))

# -------------------------
# 4. MobileNetV2 architecture
# -------------------------

model = mobilenet_v2(weights=None, num_classes=100)

# CIFAR-100 images are 32x32.
# Use stride 1 in the initial convolution instead of ImageNet's stride 2.
model.features[0][0].stride = (1, 1)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
quantizable_layers = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, (nn.Conv2d, nn.Linear))
]

quantizable_weights = sum(
    module.weight.numel()
    for _, module in quantizable_layers
)

print("\nModel configuration:")
print("  Architecture: MobileNetV2")
print("  Classes:", 100)
print("  Total parameters:", total_params)
print("  Conv2d/Linear layers:", len(quantizable_layers))
print("  Conv2d/Linear weights:", quantizable_weights)
print("  Device:", DEVICE)

# -------------------------
# 5. Optimizer and scheduler
# -------------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# -------------------------
# 6. Training and evaluation
# -------------------------

def run_epoch(model, loader, training):
    model.train(training)

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total, correct, total


history = []
best_val_accuracy = -1.0
start_time = time.time()

print("\n" + "=" * 65)
print("STARTING MOBILENETV2 TRAINING")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    train_loss, train_acc, _, _ = run_epoch(
        model, train_loader, training=True
    )

    val_loss, val_acc, val_correct, val_total = run_epoch(
        model, val_loader, training=False
    )

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_correct": val_correct,
        "val_total": val_total,
        "epoch_time_seconds": epoch_time
    }

    history.append(record)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    # Save best validation checkpoint.
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_accuracy": best_val_accuracy,
            "val_loss": val_loss,
            "seed": SEED,
            "architecture": "mobilenet_v2_cifar100_cifar32",
            "total_parameters": total_params,
            "quantizable_layers": len(quantizable_layers),
            "quantizable_weights": quantizable_weights,
            "mean": MEAN,
            "std": STD
        }, BEST_PATH)

        best_marker = "  <-- BEST"
    else:
        best_marker = ""

    # Save resumable latest checkpoint each epoch.
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_accuracy": best_val_accuracy,
        "history": history,
        "seed": SEED
    }, LATEST_PATH)

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train: {train_acc:6.2f}% | "
        f"Val: {val_acc:6.2f}% | "
        f"Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.6f} | "
        f"Time: {epoch_time:.1f}s"
        f"{best_marker}"
    )

total_training_time = time.time() - start_time

# -------------------------
# 7. Verify best checkpoint
# -------------------------

checkpoint = torch.load(
    BEST_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

final_val_loss, final_val_acc, final_val_correct, final_val_total = (
    run_epoch(model, val_loader, training=False)
)

# -------------------------
# 8. Save training configuration
# -------------------------

training_summary = {
    "architecture": "MobileNetV2",
    "dataset": "CIFAR-100",
    "seed": SEED,
    "epochs_requested": EPOCHS,
    "best_epoch": checkpoint["epoch"],
    "best_validation_accuracy": checkpoint["best_val_accuracy"],
    "verified_validation_accuracy": final_val_acc,
    "verified_validation_correct": final_val_correct,
    "verified_validation_total": final_val_total,
    "total_parameters": total_params,
    "conv_linear_layers": len(quantizable_layers),
    "conv_linear_weights": quantizable_weights,
    "batch_size": BATCH_SIZE,
    "optimizer": "SGD",
    "learning_rate_initial": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "CosineAnnealingLR",
    "training_time_seconds": total_training_time,
    "checkpoint": str(BEST_PATH)
}

with open(CONFIG_DIR / "training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)

# -------------------------
# 9. Checkpoint integrity
# -------------------------

sha256 = hashlib.sha256()

with open(BEST_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("\n" + "=" * 65)
print("MOBILENETV2 TRAINING COMPLETED")
print("=" * 65)

print("Best epoch:", checkpoint["epoch"])
print(f"Best validation accuracy: {checkpoint['best_val_accuracy']:.2f}%")
print(
    f"Verified validation accuracy: "
    f"{final_val_acc:.2f}% ({final_val_correct}/{final_val_total})"
)
print("Total parameters:", total_params)
print("Conv2d/Linear layers:", len(quantizable_layers))
print("Conv2d/Linear weights:", quantizable_weights)
print(f"Training time: {total_training_time / 60:.2f} minutes")
print("Best checkpoint:", BEST_PATH)
print("Latest checkpoint:", LATEST_PATH)
print("Training history:", HISTORY_PATH)
print("Best checkpoint SHA256:", sha256.hexdigest())

print("\nOfficial test set has NOT been used for model selection.")
print("We will evaluate it separately after validating the checkpoint.")


Split verified successfully.
Training samples: 45000
Validation samples: 5000
Dataset loaders ready:
  Training: 45000
  Validation: 5000
  Official test: 10000

Model configuration:
  Architecture: MobileNetV2
  Classes: 100
  Total parameters: 2351972
  Conv2d/Linear layers: 53
  Conv2d/Linear weights: 2317760
  Device: cuda

STARTING MOBILENETV2 TRAINING
Epoch 001/150 | Train:   7.26% | Val:  10.80% | Loss: 3.7342 | LR: 0.099989 | Time: 23.2s  <-- BEST
Epoch 002/150 | Train:  14.11% | Val:  16.82% | Loss: 3.4131 | LR: 0.099956 | Time: 16.1s  <-- BEST
Epoch 003/150 | Train:  18.02% | Val:  19.30% | Loss: 3.2995 | LR: 0.099901 | Time: 16.0s  <-- BEST
Epoch 004/150 | Train:  21.78% | Val:  20.28% | Loss: 3.2467 | LR: 0.099825 | Time: 15.5s  <-- BEST
Epoch 005/150 | Train:  25.72% | Val:  23.50% | Loss: 3.0765 | LR: 0.099726 | Time: 16.5s  <-- BEST
Epoch 006/150 | Train:  28.76% | Val:  28.14% | Loss: 2.8869 | LR: 0.099606 | Time: 16.0s  <-- BEST
Epoch 007/150 | Train:  31.53% | Val:  3